 # LangChain uses the function's name, type hints, and docstring to create a tool that an LLM can be given

In [12]:
def safe_eval(expression: str) -> float:
    """Safely evaluate basic arithmetic expressions."""
    parsed = sp.sympify(expression)

    # Allow only numeric arithmetic expressions
    if parsed.free_symbols or parsed.is_number is not True:
        raise ValueError("Only numeric expressions are allowed.")

    if parsed.has(sp.Function):
        raise ValueError("Functions are not allowed.")

    return float(parsed.evalf())
from langchain_core.tools import tool
import sympy as sp

@tool
def word_count(text:str) -> int:
    """Count the number of words in some text"""
    return len(text.split())

@tool
def calculator(expression:str) -> float:
    """Evaluate an arithmetic expression, e.g. '9 * 10'."""
    return safe_eval(expression) #never eval model input

TOOLS = [word_count , calculator]
#Create the agent - one line
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

local_model = "ollama:dagbs/qwen2.5-coder-7b-instruct-abliterated"

agent = create_agent(
    model=local_model,
    tools=TOOLS)
#Run it and read the trace 

goal = "How many words in 'the quick brown fox jumps over""the lazy dog', and that count times 10?"
result = agent.invoke({"messages": [{"role" : "user" , "content": goal}]})
print(result["messages"][-1].content)

<tools>
{
  "name": "count_words",
  "arguments": {
    "text": "the quick brown fox jumps over the lazy dog"
  }
}
</tools>
<tools>
{
  "name": "calculator",
  "arguments": {
    "expression": "10 * {count_words_result}"
  }
}
</tools>
